In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pywt

## Config

In [2]:
IN_PATH  = "../data/NF-UNSW-NB15-v3.csv"
label_col = 'Attack'
wavelet_name = 'cmor3.5-1.0'

In [3]:
#Load the data
df = pd.read_csv(IN_PATH)

In [4]:
print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (2365424, 55)
Columns:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_D

In [5]:
print("\nColumn info:")
print(df.info())


Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2365424 entries, 0 to 2365423
Data columns (total 55 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   FLOW_START_MILLISECONDS      int64  
 1   FLOW_END_MILLISECONDS        int64  
 2   IPV4_SRC_ADDR                object 
 3   L4_SRC_PORT                  int64  
 4   IPV4_DST_ADDR                object 
 5   L4_DST_PORT                  int64  
 6   PROTOCOL                     int64  
 7   L7_PROTO                     float64
 8   IN_BYTES                     int64  
 9   IN_PKTS                      int64  
 10  OUT_BYTES                    int64  
 11  OUT_PKTS                     int64  
 12  TCP_FLAGS                    int64  
 13  CLIENT_TCP_FLAGS             int64  
 14  SERVER_TCP_FLAGS             int64  
 15  FLOW_DURATION_MILLISECONDS   int64  
 16  DURATION_IN                  int64  
 17  DURATION_OUT                 int64  
 18  MIN_TTL                     

In [6]:
print("\nMissing values per column:")
print(df.isna().sum())


Missing values per column:
FLOW_START_MILLISECONDS            0
FLOW_END_MILLISECONDS              0
IPV4_SRC_ADDR                      0
L4_SRC_PORT                        0
IPV4_DST_ADDR                      0
L4_DST_PORT                        0
PROTOCOL                           0
L7_PROTO                           0
IN_BYTES                           0
IN_PKTS                            0
OUT_BYTES                          0
OUT_PKTS                           0
TCP_FLAGS                          0
CLIENT_TCP_FLAGS                   0
SERVER_TCP_FLAGS                   0
FLOW_DURATION_MILLISECONDS         0
DURATION_IN                        0
DURATION_OUT                       0
MIN_TTL                            0
MAX_TTL                            0
LONGEST_FLOW_PKT                   0
SHORTEST_FLOW_PKT                  0
MIN_IP_PKT_LEN                     0
MAX_IP_PKT_LEN                     0
SRC_TO_DST_SECOND_BYTES        63425
DST_TO_SRC_SECOND_BYTES            0
RETRANSMIT

In [7]:
print("\nDescriptive statistics for numeric columns:")
print(df.describe())


Descriptive statistics for numeric columns:


c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


       FLOW_START_MILLISECONDS  FLOW_END_MILLISECONDS   L4_SRC_PORT  \
count             2.365424e+06           2.365424e+06  2.365424e+06   
mean              1.423141e+12           1.423141e+12  3.266052e+04   
std               1.144853e+09           1.144853e+09  1.915897e+04   
min               1.421927e+12           1.421927e+12  0.000000e+00   
25%               1.421951e+12           1.421951e+12  1.592700e+04   
50%               1.424221e+12           1.424221e+12  3.281500e+04   
75%               1.424242e+12           1.424242e+12  4.910000e+04   
max               1.424263e+12           1.424263e+12  6.553500e+04   

        L4_DST_PORT      PROTOCOL      L7_PROTO      IN_BYTES       IN_PKTS  \
count  2.365424e+06  2.365424e+06  2.365424e+06  2.365424e+06  2.365424e+06   
mean   1.114841e+04  8.730817e+00  2.353191e+01  4.413591e+03  3.528169e+01   
std    1.839424e+04  6.384764e+00  2.126820e+01  6.769945e+04  7.839349e+01   
min    0.000000e+00  0.000000e+00  0.000000e

In [8]:
SAMPLES_PER_ATTACK = 200

In [9]:
attack_types = df['Attack'].unique()

In [10]:
sampled_dfs = []

for attack in attack_types:
    attack_df = df[df['Attack'] == attack]
    sample_size = min(len(attack_df), SAMPLES_PER_ATTACK)
    sampled = attack_df.sample(n=sample_size, random_state=42)
    sampled_dfs.append(sampled)

In [11]:
sampled_df = pd.concat(sampled_dfs, ignore_index=True)

In [12]:
print("Sampled dataframe shape:", sampled_df.shape)
print(sampled_df['Attack'].value_counts())

Sampled dataframe shape: (1958, 55)
Attack
Benign            200
Fuzzers           200
Exploits          200
Backdoor          200
Reconnaissance    200
Generic           200
DoS               200
Shellcode         200
Analysis          200
Worms             158
Name: count, dtype: int64


In [13]:
numeric_cols = sampled_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [14]:
exclude_cols = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS']
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]

In [15]:
scales = np.arange(1, 128)
wavelet_name = 'cmor3.5-1'
sampling_interval = 1

In [16]:
summary_features = []

In [17]:
for attack_type, group in sampled_df.groupby('Attack'):

    numeric_data = group[numeric_cols].astype(float).values
    signal = numeric_data.flatten()
    
    signal = (signal - np.mean(signal)) / (np.std(signal) + 1e-12)
    
    coefficients, freqs = pywt.cwt(signal, scales, wavelet_name)
    
    magnitude = np.abs(coefficients)
    phase = np.angle(coefficients)
    
    meanMagnitude = np.mean(magnitude)
    meanPhase = np.mean(np.abs(phase))
    
    summary_features.append({
        'Attack': attack_type,
        'meanMagnitude': meanMagnitude,
        'meanPhase': meanPhase
    })

In [18]:
import os

In [19]:
df_cwt = pd.DataFrame(summary_features)

In [20]:
print(df_cwt)

           Attack  meanMagnitude  meanPhase
0        Analysis       0.354111   1.570778
1        Backdoor            NaN        NaN
2          Benign            NaN        NaN
3             DoS            NaN        NaN
4        Exploits            NaN        NaN
5         Fuzzers            NaN        NaN
6         Generic            NaN        NaN
7  Reconnaissance            NaN        NaN
8       Shellcode            NaN        NaN
9           Worms            NaN        NaN


In [43]:
os.chmod("cwt_results", 0o777)

In [44]:
df_cwt.to_csv("cwt_results")

PermissionError: [Errno 13] Permission denied: 'cwt_results'